In [1]:
# Prepare dataframe from daily rasters for prediction over the entire grid
# we use the fitted model from 3A_2

# WORKFLOW
#  Load all predictor rasters for the day
#  Stack and create a df
#  Format df to fit column requirement of the model
#  Make predictions for tmax or tmin in the dataframe
# Next code: Make a raster based on the predicted datframe

In [ ]:
#imports

import os
import gc
import glob
import numpy as np
import pandas as pd 
import xarray as xr
import rioxarray as rxr
import pyarrow as pa    
import pyarrow.parquet as pq
from tqdm import tqdm

START_YEAR, START_MONTH = 2001, 1
END_YEAR, END_MONTH = 2005, 12

MONTHLY_RASTER_TEMP = r"..\Data_Temp\Processed\Monthly_downscaling\Input_rasters"
MONTHLY_RASTER_PREC = r"..\Data\Processed\Regression_model\Monthly_Rasters"

# paths for static rasters
static_paths = {
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

#template grid
template = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")
OUT_BASE = fr"..\Data_Temp\Processed\Monthly_downscaling"

tmax_temp_dir = os.path.join(OUT_BASE, 'Temp_tmax')
tmax_dynamic_pred_dir = os.path.join(tmax_temp_dir, 'Dynamic_pred')
tmax_yearly_parquet_dir = os.path.join(tmax_temp_dir, 'Tmax_yearly_parquet')


tmin_temp_dir = os.path.join(OUT_BASE, 'Temp_tmin')
tmin_dynamic_pred_dir = os.path.join(tmin_temp_dir, 'Dynamic_pred')
tmin_yearly_parquet_dir = os.path.join(tmin_temp_dir, 'Tmin_yearly_parquet')


static_dir = os.path.join(OUT_BASE, "Static_df")

for d in [tmax_temp_dir, tmax_dynamic_pred_dir, tmax_yearly_parquet_dir, 
    static_dir, tmin_temp_dir, tmin_dynamic_pred_dir, tmin_yearly_parquet_dir]:
    if not os.path.exists(d):
        os.makedirs(d)

In [ ]:
# HELPERS

def month_cyclical(month: int):
    """Return (sin, cos) encoding for a calendar month (1–12)."""
    angle = 2 * np.pi * month / 12
    return np.sin(angle), np.cos(angle)
 
def mlabel(year: int, month: int) -> str:
    return f"{year}{month:02d}"
    
def build_monthly_paths_temp(year: int, month: int, t_variable: str) -> dict:
    """
    returns a dictionary of {feature_name: filepath} for all monthly rasters
    within the temperature folder (gridded temperature products)
    for the given year-month-variable (variable: tmax or tmin)
    """
    L = mlabel(year, month)
    R = MONTHLY_RASTER_TEMP
    return {
        #Temperature average variables
        f"chirts_{t_variable}_average": f"{R}\\chirts\\chirts_{t_variable}_average_{L}.tif",
        f"era5l_{t_variable}_average": f"{R}\\era5l\\era5l_{t_variable}_average_{L}.tif",
        f"har_{t_variable}_average": f"{R}\\har\\har_{t_variable}_average_{L}.tif",

        #Temperature secondary variables from CHIRTS 
        f"chirts_{t_variable}_max": f"{R}\\chirts\\chirts_{t_variable}_max_{L}.tif",
        f"chirts_{t_variable}_min": f"{R}\\chirts\\chirts_{t_variable}_min_{L}.tif",
        f"chirts_{t_variable}_std": f"{R}\\chirts\\chirts_{t_variable}_std_{L}.tif",
        f"chirts_dtr_std": f"{R}\\chirts\\chirts_dtr_std_{L}.tif", 

        #Insitu 
        f"ik_{t_variable}_average": f"..\\Data_Temp\\Datasets\\Kriging_monthly\\{t_variable}\\{t_variable}_monthly_{L}.tif"     
    }

def build_monthly_paths_prec(year: int, month: int) -> dict:
    """
    returns a dictionary of {feature_name: filepath} for all monthly rasters
    within the precipitation folder (secondary meteorological variables and evi)
    for the given year-month
    """
    L = mlabel(year, month)
    R = MONTHLY_RASTER_PREC
    
    return {
        #GLDAS weather variables
        'gldas_air_temp':   f"{R}\\gldas_air_temp\\gldas_air_temp_monthly_mean_{L}.tif",
        'gldas_lwdown': f"{R}\\gldas_lwdown\\gldas_lwdown_monthly_sum_{L}.tif",
        'gldas_specific_humidity': f"{R}\\gldas_specific_humidity\\gldas_specific_humidity_monthly_mean_{L}.tif",
        'gldas_surface_pressure': f"{R}\\gldas_surface_pressure\\gldas_surface_pressure_monthly_mean_{L}.tif",
        'gldas_swdown': f"{R}\\gldas_swdown\\gldas_swdown_monthly_sum_{L}.tif",

        # MODIS EVI
        "evi_monthly_mean":         f"{R}\\modis_evi\\evi_monthly_mean_{L}.tif",
        "evi_missing_fraction":     f"{R}\\modis_evi\\evi_missing_fraction_{L}.tif",

        # MODIS LST
        "lst_d_monthly_mean":       f"{R}\\modis_lst\\lst_d_monthly_mean_{L}.tif",
        "lst_d_missing_fraction":   f"{R}\\modis_lst\\lst_d_missing_fraction_{L}.tif",
        "lst_n_monthly_mean":       f"{R}\\modis_lst\\lst_n_monthly_mean_{L}.tif",
        "lst_n_missing_fraction":   f"{R}\\modis_lst\\lst_n_missing_fraction_{L}.tif",
        "lst_diff_monthly_mean":    f"{R}\\modis_lst\\lst_diff_monthly_mean_{L}.tif",
        "lst_diff_missing_fraction":f"{R}\\modis_lst\\lst_diff_missing_fraction_{L}.tif"
    }

In [4]:
# 1. Static feature dataframe -------------------------------------

static_csv = os.path.join(static_dir, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in static_paths.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        if "spatial_ref" in da.coords:
            da = da.drop_vars("spatial_ref")
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
print(f"Static df: {len(static_df):,} pixels")

Loading cached static df...
Static df: 484,619 pixels


In [5]:
# 2 run the main month loop

month_range = pd.period_range(
    start = f"{START_YEAR}-{START_MONTH:02d}",
    end = f"{END_YEAR}-{END_MONTH:02d}",
    freq = "M"
)

for t_variable in ['tmax', 'tmin']:
    print(f"\n ============ Processing {len(month_range)} months for {t_variable.upper()} =========")
    for period in tqdm(month_range, desc = "Monthly feature extraction"):
        year = period.year
        month = period.month
        label = mlabel(year, month)
        
        # Write directly to the dynamic pred directory to match Cell 5
        if t_variable == 'tmax':
            pred_out = os.path.join(tmax_dynamic_pred_dir, f"tmax_ds_{label}.parquet")
        else:
            pred_out = os.path.join(tmin_dynamic_pred_dir, f"tmin_ds_{label}.parquet")

        if os.path.exists(pred_out):
            continue

        paths_temp = build_monthly_paths_temp(year, month, t_variable)
        paths_prec = build_monthly_paths_prec(year, month)
        paths = paths_temp | paths_prec
        
        aligned = []
        missing_vars = []

        for name, path in paths.items():
            if not os.path.exists(path):
                missing_vars.append(name)
                continue
            da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
            da = da.rio.reproject_match(template).squeeze("band", drop = True)
            da.name = name
            aligned.append(da.load())
            da.close()

        if missing_vars:
            tqdm.write(f" [{label}] missing rasters: {missing_vars}")
        
        if not aligned:
            tqdm.write(f" [{label}] no rasters found - skipping")
            continue

        dynamic_xr = xr.merge(aligned, compat='override')
        dynamic_df = dynamic_xr.to_dataframe().reset_index()

        # variable without rasters gets Nan Column
        for name in missing_vars:
            dynamic_df[name] = np.nan

        del aligned, dynamic_xr
        gc.collect()

        #merge with static df
        monthly_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = "inner")

        #Add temporal features - cyclical month columns
        sin_m, cos_m = month_cyclical(month)
        monthly_grid['month_sin'] = sin_m
        monthly_grid['month_cos'] = cos_m
        monthly_grid['year'] = year
        monthly_grid['month'] = month

        # Write directly to Parquet - saves time and space!
        monthly_grid.to_parquet(pred_out, index=False)

        del monthly_grid
        gc.collect()


 ============ Processing 60 months for TMAX =========


Monthly feature extraction: 100%|██████████| 60/60 [00:00<00:00, 3703.14it/s]



 ============ Processing 60 months for TMIN =========


Monthly feature extraction: 100%|██████████| 60/60 [00:00<00:00, 9207.12it/s]


In [6]:
# 3 aggregate monthly parquets to yearly parquet 

def aggregate_to_parquet(monthly_dir: str, parquet_dir: str, t_variable: str) -> str:
    """
    Read monthly Parquets year by year, write to a single yearly parquet.
    Yields a list of yearly parquets.
    """
    parquet_paths = []

    for year in range(START_YEAR, END_YEAR + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        monthly_periods = pd.period_range(
            start = f"{year}-01", end = f"{year}-12", freq = "M"
        )
        expected = [
            os.path.join(monthly_dir, f"{t_variable}_ds_{mlabel(p.year, p.month)}.parquet")
            for p in monthly_periods
        ]
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            print(f"  {t_variable} |  {year} - no monthly files found, skipping")
            continue

        writer = None
        for fpath in found:
            chunk = pd.read_parquet(fpath)
            table = pa.Table.from_pandas(chunk)
            if writer is None:
                writer = pq.ParquetWriter(yearly_path, table.schema)
            writer.write_table(table)
        
        if writer:
            writer.close()
            parquet_paths.append(yearly_path)
            print(f"{year} - {len(found)} months written to parquet")
    
    if not parquet_paths:
        print(f"   No data in parquet paths - final df not written")
        return None
        
    print(f"\n  [{t_variable}] Yearly files aggregated successfully in {parquet_dir}.")
    print(f"  Load entire dataset memory-safely using: pd.read_parquet(r'{parquet_dir}')")
    
    return parquet_paths

print("\n ================Aggregating to yearly parquet files================")

tmax_pred_final_paths = aggregate_to_parquet(monthly_dir = tmax_dynamic_pred_dir, 
    parquet_dir = tmax_yearly_parquet_dir, 
    t_variable= 'tmax')

tmin_pred_final_paths = aggregate_to_parquet(monthly_dir = tmin_dynamic_pred_dir, 
    parquet_dir = tmin_yearly_parquet_dir, 
    t_variable= 'tmin')


 ================Aggregating to yearly parquet files================
2001 - 12 months written to parquet
2002 - 12 months written to parquet
2003 - 12 months written to parquet
2004 - 12 months written to parquet
2005 - 12 months written to parquet

  [tmax] Yearly files aggregated successfully in ..\Data_Temp\Processed\Monthly_downscaling\Temp_tmax\Tmax_yearly_parquet.
  Load entire dataset memory-safely using: pd.read_parquet(r'..\Data_Temp\Processed\Monthly_downscaling\Temp_tmax\Tmax_yearly_parquet')
2001 - 12 months written to parquet
2002 - 12 months written to parquet
2003 - 12 months written to parquet
2004 - 12 months written to parquet
2005 - 12 months written to parquet

  [tmin] Yearly files aggregated successfully in ..\Data_Temp\Processed\Monthly_downscaling\Temp_tmin\Tmin_yearly_parquet.
  Load entire dataset memory-safely using: pd.read_parquet(r'..\Data_Temp\Processed\Monthly_downscaling\Temp_tmin\Tmin_yearly_parquet')
